# Case-study LCA plots
Stacked contribution panels inspired by the reference figure: one panel per impact category, one bar per database, and a shared legend. A separate figure is produced for each country/scenario pair actually present in the results.

Colors, hatches, contributor names and units come from `mapping.py`. Results are expressed per **kg NH₃**, using annual production `T_DAY × 365 × 1000`; set the production below to the value used when generating the results. Database years are read from database names, not the foreground assessment year.

Start with a fresh kernel so the sequential MKL threading setting takes effect before numerical libraries load.


In [ ]:
# Set before NumPy/pandas imports; restart the kernel before running this notebook.
# Matches the sequential MKL setting used by the case-study calculation script.
import os
os.environ["MKL_THREADING_LAYER"] = "SEQUENTIAL"

from pathlib import Path
import math
import re
import textwrap
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from IPython.display import display

from config import FILE_PATH_CASE_STUDIES_LCA, T_DAY
from mapping import (color_dict_env, hex_dict_env, name_dict_fig,
                     contribution_mapping_system, dict_units, column_dict_categories)

RESULTS_PATH = Path(FILE_PATH_CASE_STUDIES_LCA)
OUTPUT_DIR = Path("figs/case-study-plots")
DATABASES = None  # None = all available; or a list of exact database names
COUNTRIES = None  # None = all available; or a list of country names
SCENARIOS = None  # None = all available; or e.g. ["off_grid"]
CATEGORIES = None  # None = all identifiable categories; or their mapped names
ANNUAL_PRODUCTION_KG = T_DAY * 365 * 1000
N_COLUMNS = 5
SAVE_FIGURES = True
DPI = 180


## Load and identify impact categories
Some older result files store only the final element of an LCIA method tuple. That label is not always unique. Ambiguous labels are listed below and excluded: their original categories cannot be recovered reliably from these files. Regenerated data should retain full method tuples or unique category names. No missing database is substituted and missing category/database combinations are marked explicitly.


In [ ]:
def resolve_category(raw):
    """Match full method tuples, category names, or unambiguous indicator names."""
    if isinstance(raw, tuple) and raw in dict_units:
        return raw[-2], dict_units[raw]
    matches = [(method, unit) for method, unit in dict_units.items()
               if raw == method[-2]]
    if not matches:
        matches = [(method, unit) for method, unit in dict_units.items()
                   if raw == method[-1]]
    identities = {(method[-2], unit) for method, unit in matches}
    if len(identities) == 1:
        return next(iter(identities))
    return None


def prepare_results(frame, annual_production_kg):
    if not np.isfinite(annual_production_kg) or annual_production_kg <= 0:
        raise ValueError("Annual NH3 production must be finite and positive.")
    data = frame.reset_index()
    value_col = "results" if "results" in data else "multi_energy_system"
    required = {"country", "scenario", "db_name", "category", "contributor", value_col}
    if not required.issubset(data.columns):
        raise ValueError(f"Missing result columns: {sorted(required - set(data.columns))}")
    raw_categories = data["category"].unique().tolist()
    resolved = {raw: resolve_category(raw) for raw in raw_categories}
    excluded = [raw for raw, result in resolved.items() if result is None]
    if excluded:
        warnings.warn("Excluded ambiguous or unmapped category labels: " + repr(excluded))
    data = data[data["category"].map(lambda raw: resolved[raw] is not None)].copy()
    data["unit"] = data["category"].map(lambda raw: resolved[raw][1])
    data["category"] = data["category"].map(lambda raw: resolved[raw][0])
    data["contributor"] = data["contributor"].map(
        lambda value: contribution_mapping_system.get(value, value))
    data["impact"] = pd.to_numeric(data[value_col], errors="raise") / annual_production_kg
    if not np.isfinite(data["impact"]).all():
        raise ValueError("LCA contributions contain missing or non-finite values.")
    # Do not accidentally sum separate foreground assessment years.
    if "year" in data and (data.groupby(["country", "scenario", "db_name"])["year"].nunique() > 1).any():
        raise ValueError("Multiple assessment years per case: select one year before plotting.")
    return data, excluded

raw_results = pd.read_pickle(RESULTS_PATH)
data, excluded_categories = prepare_results(raw_results, ANNUAL_PRODUCTION_KG)
for column, selection in [("db_name", DATABASES), ("country", COUNTRIES),
                          ("scenario", SCENARIOS), ("category", CATEGORIES)]:
    if selection is not None:
        missing = set(selection) - set(data[column])
        if missing:
            raise ValueError(f"Requested {column} values unavailable: {sorted(missing)}")
        data = data[data[column].isin(selection)]
if data.empty:
    raise ValueError("No identifiable LCA results match the selected filters.")
display(data[["country", "scenario", "db_name"]].drop_duplicates().reset_index(drop=True))
print(f"{data.category.nunique()} impact categories; {len(excluded_categories)} excluded labels.")


In [ ]:
def database_labels(databases):
    years = [re.search(r"_(20\d{2})(?:_|$)", db) for db in databases]
    labels = [match.group(1) if match else db for db, match in zip(databases, years)]
    if len(set(labels)) != len(labels):
        labels = [db.replace("ecoinvent-", "").replace("_", " ") for db in databases]
    return [textwrap.fill(label, 24) for label in labels]


def plot_case(case, country, scenario, database_order=None):
    databases = ([db for db in database_order if db in set(case.db_name)]
                 if database_order is not None else sorted(case.db_name.unique()))
    categories = sorted(case.category.unique())
    contributors = sorted(case.contributor.unique())
    # Retain all-zero panels, but keep inactive contributors out of the legend.
    contributors = [c for c in contributors if case.loc[case.contributor == c, "impact"].ne(0).any()]
    fallback = plt.get_cmap("tab20")
    colors = {c: color_dict_env.get(c, fallback(i % 20)) for i, c in enumerate(contributors)}
    hatches = {c: hex_dict_env.get(c, "") for c in contributors}
    unmapped = [c for c in contributors if c not in color_dict_env]
    if unmapped:
        warnings.warn(f"Using fallback colors for unmapped contributors: {unmapped}")
    ncols = min(N_COLUMNS, len(categories))
    if ncols < 1:
        raise ValueError("N_COLUMNS must be positive.")
    nrows = math.ceil(len(categories) / ncols)
    legend_rows = math.ceil((len(contributors) + 1) / 3)
    legend_height = 0.28 * legend_rows + 0.35
    height = 3.25 * nrows + 1.05 + legend_height
    with plt.rc_context({"font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
                         "hatch.linewidth": 0.5, "axes.spines.top": True,
                         "axes.spines.right": True}):
        fig, axes = plt.subplots(nrows, ncols, figsize=(3.25 * ncols, height), squeeze=False)
        for ax, category in zip(axes.flat, categories):
            selected = case[case.category == category]
            present = set(selected.db_name)
            table = selected.pivot_table(index="db_name", columns="contributor", values="impact", aggfunc="sum")
            table = table.reindex(index=databases, columns=contributors).fillna(0)
            positive, negative = np.zeros(len(databases)), np.zeros(len(databases))
            x = np.arange(len(databases))
            for contributor in contributors:
                values = table[contributor].to_numpy()
                ax.bar(x, values, bottom=np.where(values >= 0, positive, negative), width=0.64,
                       color=colors[contributor], hatch=hatches[contributor],
                       edgecolor="black", linewidth=0.25)
                positive += np.maximum(values, 0)
                negative += np.minimum(values, 0)
            totals = table.sum(axis=1).to_numpy()
            valid = np.array([db in present for db in databases])
            ax.plot(x[valid], totals[valid], "kD", markersize=3, linestyle="none")
            for i, exists in enumerate(valid):
                if not exists:
                    ax.text(i, 0.05, "No data", transform=ax.get_xaxis_transform(), ha="center", fontsize=8)
            ax.axhline(0, color="black", linewidth=0.6)
            if not positive.any() and not negative.any():
                ax.set_ylim(0, 1)
                ax.text(0.5, 0.5, "All contributions are zero", transform=ax.transAxes,
                        ha="center", fontsize=9)
            else:
                lower, upper = min(0.0, negative.min()), max(0.0, positive.max())
                padding = (upper - lower) * 0.15
                ax.set_ylim(lower - padding if lower < 0 else 0, upper + padding)
            ax.set_xticks(x, database_labels(databases))
            ax.set_xlim(-0.65, len(databases) - 0.35)
            title = category
            if category == "climate change":
                title += " — GWP 100a"
            ax.set_title(textwrap.fill(title, 29), pad=9)
            ax.set_ylabel(selected.unit.iloc[0] + " / kg NH$_3$")
            ax.ticklabel_format(axis="y", style="sci", scilimits=(-3, 4), useMathText=True)
            ax.tick_params(direction="out", length=3)
        for ax in list(axes.flat)[len(categories):]:
            ax.set_visible(False)
        handles = [Patch(facecolor=colors[c], hatch=hatches[c], edgecolor="black", linewidth=0.3,
                         label=name_dict_fig.get(c, c)) for c in contributors]
        handles.append(Line2D([], [], color="black", marker="D", linestyle="none", markersize=4, label="Net total"))
        fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.015), ncol=3, frameon=False, fontsize=9)
        fig.suptitle(f"{country} | {scenario.replace('_', ' ').replace('-', ' ').title()}\nAmmonia production — LCA contributions", fontsize=16, y=0.985)
        fig.tight_layout(rect=(0, legend_height / height, 1, 1 - 0.8 / height), h_pad=2.5, w_pad=2)
    return fig


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_files = []
for (country, scenario), case in data.groupby(["country", "scenario"], sort=True):
    fig = plot_case(case, country, scenario, DATABASES)
    if SAVE_FIGURES:
        stem = re.sub(r"[^A-Za-z0-9_-]+", "_", f"{country}_{scenario}").strip("_")
        for extension in ("png", "pdf"):
            path = OUTPUT_DIR / f"{stem}.{extension}"
            fig.savefig(path, dpi=DPI, bbox_inches="tight", facecolor="white")
            output_files.append(path)
    plt.show()
    plt.close(fig)
for path in output_files:
    print(path)
